  Terminal 1: Gazebo / task board

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  export DBX_CONTAINER_MANAGER=docker
  distrobox enter -r aic_eval -- /entrypoint.sh \
    spawn_task_board:=true \
    task_board_x:=0.3 task_board_y:=-0.1 task_board_z:=1.2 \
    task_board_roll:=0.0 task_board_pitch:=0.0 task_board_yaw:=0.785 \
    sfp_mount_rail_0_present:=tr• I checked the existing branches and the current Isaac setup. If I have to choose one lane right now, I
ue sfp_mount_rail_0_translation:=-0.08 \
    sc_mount_rail_0_present:=true sc_mount_rail_0_translation:=-0.09 \
    nic_card_mount_0_present:=true nic_card_mount_0_translation:=0.005 \
    sc_port_0_present:=true sc_port_0_translation:=-0.04 \
    spawn_cable:=true \
    cable_type:=sfp_sc_cable \
    attach_cable_to_gripper:=true \
    ground_truth:=true \
    start_aic_engine:=true \
    shutdown_on_aic_engine_exit:=false \
    skip_model_ready:=true \
    skip_ready_simulator:=true



  Terminal 2: recorder

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  unset CONDA_SUBDIR
  bash rangers_teleop/bash/gazebo_record.sh --session-name teleop_session_001

  Terminal 3: Cartesian teleop

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  unset CONDA_SUBDIR
  bash rangers_teleop/bash/gazebo_cartesian_teleop.sh

  Terminal 4: episode control
  Start trial:

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  bash rangers_teleop/bash/gazebo_episode_start.sh

  Stop trial:

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  bash rangers_teleop/bash/gazebo_episode_stop.sh


### Agent A

• Yes, with one correction: 80% cheatcode / 20% teleop is a reasonable starting target, but it is not the
  real success criterion.

  The real success case is bucket coverage plus replay quality. I’ve set that up in aic-rangers-isaac-
  training/rangers-data/README.md and linked it from aic-rangers-isaac-training/rangers_training/
  README.md. The data lane is now explicitly separated from the training lane.

  What I want from this data lane is:

  - scripted cheatcode data provides most of the volume
  - teleop provides the high-value correction/recovery behavior
  - both SFP and SC are covered
  - we can intentionally produce clean_success, perturbed_success, near_miss, recovery, and hard_failure
  - you replay samples from each bucket and confirm they are visually sane

  So yes, you should absolutely peek at replays. I would treat that as mandatory, not optional. Minimum
  review pass:

  - 3-5 sample episodes per bucket
  - for both SFP and SC

  If the replay looks wrong, the dataset is wrong, regardless of what the logs say.

  Next on my end, I would keep this agent on the data side:

  1. mine Enrique’s cheatcode/recorder branch work selectively,
  2. build a repeatable scripted data-generation path,
  3. make the generated data easy to replay and audit before we feed it back into ACT/flow.

  I would not switch this lane to Isaac RL first. The current ACT 38.5 result is useful, but the Isaac RL
  stack is PPO-native, so “continue ACT inside Isaac” is not the quick win here. Data quality is still
  the higher-leverage move.


### Agent B - Teleop

### Agent C - Eval Pipeline


  Current split is:

  - Hugging Face: model storage and recovery only. That is where the ACT checkpoints live now, since you deleted local
    rangers_training/outputs.
  - Gazebo evaluator: actual policy evaluation. The scores like 38.5, 71.55, etc. come from local runs in aic-rangers/
    aic_eval_runs/.../results/scoring.yaml.
  - W&B: training curves and run logs, not the benchmark score source of truth.

  So the evaluation path is:

  1. download ACT checkpoint from HF
  2. run it locally in Gazebo through the evaluator
  3. read the AIC score from the local scoring file

  There is no HF “evaluation dashboard” in the current setup. If you want one, the sensible options are:

  - push evaluation summaries into W&B
  - or write a small local leaderboard CSV/Markdown under aic_eval_runs
  - or upload evaluator artifacts to a separate HF dataset/repo later

  Right now, the benchmark truth is local Gazebo evaluator output, not HF.
